# Technical Note 39 — Machine Learning in Derivatives

Code behind every number in the chapter. **NumPy only** — no autodiff library.
That is deliberate: the gradient recursions in §39.6 are written out here as code,
so the implementation and the text explain each other.

| Section | What it reproduces |
|---|---|
| 1 | Heston pricer by Gauss–Legendre quadrature, implied-vol inversion |
| 2 | MLP with reverse-mode gradients written out by hand |
| 3 | The pricing surrogate (Example 2.1) — 1,345 parameters |
| 4 | Outside the training box (Example 3.1, Figure 1) |
| 5 | The pricer bug (Business Snapshot 39.1) |
| 6 | Differential learning (Example 5.1) |
| 7 | Deep hedging with CVaR (Example 7.1) — 1,217 parameters |
| 8 | Seed dependence (Example 12.2, Figure 4) |

Set `FAST = True` for a few minutes end-to-end; `False` reproduces the chapter's
sample sizes and takes considerably longer. Last digits depend on seed and BLAS.

In [ ]:
import numpy as np, time, math
from scipy.special import roots_legendre
from scipy.optimize import brentq
from scipy.stats import norm
import matplotlib.pyplot as plt

FAST = True
rng_global = np.random.default_rng(0)
print('FAST =', FAST)

## 1. The Heston pricer, and the quadrature that has to converge

The surrogate can be no better than its labels (§39.4), so the pricer comes first.
`n_nodes` is exposed on purpose — §5 uses it to reproduce the bug.

In [ ]:
def heston_cf(u, S, T, r, q, v0, theta, kappa, sigma_v, rho):
    """Characteristic function of log spot, in the numerically stable form."""
    iu = 1j * u
    b = kappa - rho * sigma_v * iu
    d = np.sqrt(b**2 + sigma_v**2 * (iu + u**2))
    g = (b - d) / (b + d)                      # |g| < 1 branch: no log wrapping
    e = np.exp(-d * T)
    C = (kappa * theta / sigma_v**2) * ((b - d) * T - 2.0 * np.log((1 - g * e) / (1 - g)))
    D = (v0 / sigma_v**2) * (b - d) * (1 - e) / (1 - g * e)
    return np.exp(iu * (np.log(S) + (r - q) * T) + C + D)


def heston_call(S, K, T, r, q, v0, theta, kappa, sigma_v, rho, n_nodes=600, upper=200.0):
    """European call by Gil-Pelaez inversion on a Gauss-Legendre grid."""
    x, w = roots_legendre(n_nodes)
    u = 0.5 * upper * (x + 1.0)                 # map [-1,1] -> [0, upper]
    w = 0.5 * upper * w
    lnK = np.log(K)
    cf_u = heston_cf(u, S, T, r, q, v0, theta, kappa, sigma_v, rho)
    cf_ui = heston_cf(u - 1j, S, T, r, q, v0, theta, kappa, sigma_v, rho)
    denom = heston_cf(-1j, S, T, r, q, v0, theta, kappa, sigma_v, rho).real
    P1 = 0.5 + (w * (np.exp(-1j * u * lnK) * cf_ui / (1j * u * denom)).real).sum() / np.pi
    P2 = 0.5 + (w * (np.exp(-1j * u * lnK) * cf_u / (1j * u)).real).sum() / np.pi
    return S * np.exp(-q * T) * P1 - K * np.exp(-r * T) * P2


def bs_call(S, K, T, r, q, sig):
    if sig <= 0 or T <= 0:
        return max(S * np.exp(-q * T) - K * np.exp(-r * T), 0.0)
    d1 = (np.log(S / K) + (r - q + 0.5 * sig**2) * T) / (sig * np.sqrt(T))
    d2 = d1 - sig * np.sqrt(T)
    return S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def implied_vol(price, S, K, T, r, q):
    intrinsic = max(S * np.exp(-q * T) - K * np.exp(-r * T), 0.0)
    if price <= intrinsic + 1e-12:
        return np.nan
    try:
        return brentq(lambda s: bs_call(S, K, T, r, q, s) - price, 1e-4, 5.0, xtol=1e-10)
    except ValueError:
        return np.nan


# sanity: a known parameter set, and the price should be stable in the node count
base = dict(S=100.0, K=100.0, T=1.0, r=0.02, q=0.0, v0=0.04, theta=0.04,
            kappa=1.5, sigma_v=0.3, rho=-0.6)
for n in (100, 200, 600, 1200):
    print('%5d nodes  price %.10f' % (n, heston_call(n_nodes=n, **base)))

## 2. A network, with its gradients written out

Affine maps and `tanh`, per §39.6. The backward pass is the recursion
$\delta^{(\ell)} = (W^{(\ell+1)\top}\delta^{(\ell+1)}) \odot \varphi'(z^{(\ell)})$,
and `input_grad` stops one step earlier — that is what §39.6 needs for the hedging
policy, whose output at one date is an input at the next.

In [ ]:
class MLP:
    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.sizes = sizes
        self.W, self.b = [], []
        for a, c in zip(sizes[:-1], sizes[1:]):
            self.W.append(rng.normal(0, np.sqrt(1.0 / a), (a, c)))
            self.b.append(np.zeros(c))

    @property
    def n_params(self):
        return sum(w.size for w in self.W) + sum(b.size for b in self.b)

    def forward(self, x):
        """Return output and the cache needed for the backward pass."""
        a, cache = x, [x]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = a @ W + b
            a = np.tanh(z) if i < len(self.W) - 1 else z      # identity on output
            cache.append(a)
        return a, cache

    def __call__(self, x):
        return self.forward(x)[0]

    def backward(self, cache, dout):
        """Reverse mode. Returns (dW, db, dx) — dx is equation (39.x)'s input gradient."""
        dW = [None] * len(self.W)
        db = [None] * len(self.b)
        delta = dout
        for i in reversed(range(len(self.W))):
            a_prev = cache[i]
            dW[i] = a_prev.T @ delta
            db[i] = delta.sum(axis=0)
            delta = delta @ self.W[i].T
            if i > 0:
                delta = delta * (1.0 - cache[i] ** 2)          # tanh'
        return dW, db, delta

    def flat(self):
        return np.concatenate([w.ravel() for w in self.W] + [b.ravel() for b in self.b])


class Adam:
    def __init__(self, shapes, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.m = [np.zeros(s) for s in shapes]
        self.v = [np.zeros(s) for s in shapes]
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0

    def step(self, params, grads):
        self.t += 1
        for i, (p, g) in enumerate(zip(params, grads)):
            self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * g
            self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * g * g
            mh = self.m[i] / (1 - self.b1 ** self.t)
            vh = self.v[i] / (1 - self.b2 ** self.t)
            p -= self.lr * mh / (np.sqrt(vh) + self.eps)


print('surrogate  7-32-32-1 :', MLP([7, 32, 32, 1]).n_params, 'parameters')
print('policy     3-32-32-1 :', MLP([3, 32, 32, 1]).n_params, 'parameters')

## 3. The pricing surrogate — Example 2.1

The training box of §39.3. Points with less than 0.1% of spot in time value are
dropped, because an implied volatility is not defined in any useful sense for a
quote that does not exist.

In [ ]:
BOX = dict(v0=(0.01, 0.16), theta=(0.01, 0.16), kappa=(0.5, 3.0),
           sigma_v=(0.1, 1.0), rho=(-0.9, 0.0), moneyness=(0.7, 1.3), T=(0.1, 2.0))
KEYS = list(BOX)
R, Q, S0 = 0.02, 0.0, 100.0


def sample_box(n, box, seed):
    rng = np.random.default_rng(seed)
    return np.column_stack([rng.uniform(*box[k], n) for k in KEYS])


def label(X, n_nodes=600):
    """Price each row and invert to implied volatility; NaN where undefined."""
    out = np.full(len(X), np.nan)
    for i, row in enumerate(X):
        v0, th, ka, sv, rh, mny, T = row
        K = S0 * mny
        p = heston_call(S0, K, T, R, Q, v0, th, ka, sv, rh, n_nodes=n_nodes)
        intrinsic = max(S0 * np.exp(-Q * T) - K * np.exp(-R * T), 0.0)
        if p - intrinsic < 0.001 * S0:            # the 0.1%-of-spot filter
            continue
        out[i] = implied_vol(p, S0, K, T, R, Q)
    return out


N_TRAIN = 4000 if FAST else 30000
N_TEST = 1000 if FAST else 6000
t0 = time.time()
Xtr = sample_box(N_TRAIN, BOX, 1); ytr = label(Xtr)
Xte = sample_box(N_TEST, BOX, 2);  yte = label(Xte)
keep_tr, keep_te = ~np.isnan(ytr), ~np.isnan(yte)
Xtr, ytr, Xte, yte = Xtr[keep_tr], ytr[keep_tr], Xte[keep_te], yte[keep_te]
print('labeling took %.1f s; kept %d train, %d test' % (time.time() - t0, len(ytr), len(yte)))

In [ ]:
mu, sd = Xtr.mean(0), Xtr.std(0)
norm_x = lambda X: (X - mu) / sd


def train_surrogate(X, y, seed=0, epochs=None, batch=256, lr=3e-3):
    epochs = epochs or (60 if FAST else 300)
    net = MLP([X.shape[1], 32, 32, 1], seed=seed)
    opt = Adam([w.shape for w in net.W] + [b.shape for b in net.b], lr=lr)
    Xn, yv = norm_x(X), y.reshape(-1, 1)
    rng = np.random.default_rng(seed + 99)
    for ep in range(epochs):
        idx = rng.permutation(len(Xn))
        for s in range(0, len(idx), batch):
            j = idx[s:s + batch]
            out, cache = net.forward(Xn[j])
            resid = out - yv[j]
            dW, db, _ = net.backward(cache, 2.0 * resid / len(j))
            opt.step(net.W + net.b, dW + db)
    return net


t0 = time.time(); surrogate = train_surrogate(Xtr, ytr); train_s = time.time() - t0
err_bp = (surrogate(norm_x(Xte)).ravel() - yte) * 1e4
print('trained in %.1f s (%d parameters)' % (train_s, surrogate.n_params))
print('RMSE      %7.1f bp' % np.sqrt((err_bp ** 2).mean()))
print('MAE       %7.1f bp' % np.abs(err_bp).mean())
print('95th pct  %7.1f bp' % np.percentile(np.abs(err_bp), 95))
print('worst     %7.1f bp' % np.abs(err_bp).max())

In [ ]:
# the speed claim: per-price cost, quadrature against surrogate
row = Xte[0]
t0 = time.time()
for _ in range(200):
    heston_call(S0, S0 * row[5], row[6], R, Q, row[0], row[1], row[2], row[3], row[4])
quad_ms = (time.time() - t0) / 200 * 1e3

Xb = norm_x(np.repeat(row[None, :], 1000, 0))
t0 = time.time()
for _ in range(50):
    surrogate(Xb)
surr_ms = (time.time() - t0) / (50 * 1000) * 1e3
print('quadrature %.4f ms   surrogate %.6f ms   speed-up %.0fx'
      % (quad_ms, surr_ms, quad_ms / surr_ms))
print('one calibration (200 quotes x 60 evaluations): %.2f s vs %.3f s'
      % (quad_ms * 200 * 60 / 1e3, surr_ms * 200 * 60 / 1e3))

## 4. Outside the box — Example 3.1 and Figure 1

The same network, unchanged, evaluated beyond the region it was fitted on.
The failure is silent: the numbers come back the usual size.

In [ ]:
def rmse_on(box, seed):
    X = sample_box(600 if FAST else 3000, box, seed)
    y = label(X)
    k = ~np.isnan(y)
    e = (surrogate(norm_x(X[k])).ravel() - y[k]) * 1e4
    return np.sqrt((e ** 2).mean()), np.abs(e).max()

cases = {'Inside box': BOX,
         'Vol-of-vol': {**BOX, 'sigma_v': (1.0, 1.6)},
         'Maturity':   {**BOX, 'T': (2.0, 5.0)},
         'Strike':     {**BOX, 'moneyness': (1.3, 1.9)}}
res = {}
for i, (name, bx) in enumerate(cases.items()):
    res[name] = rmse_on(bx, 50 + i)
    print('%-12s RMSE %8.1f bp   worst %8.1f bp' % (name, *res[name]))

fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.bar(list(res), [v[0] for v in res.values()], color='#8fa8c8', edgecolor='#1f3b63')
ax.set_yscale('log'); ax.set_ylabel('RMSE (bp, log scale)')
for i, v in enumerate(res.values()):
    ax.text(i, v[0] * 1.15, '%.1f' % v[0], ha='center', fontsize=9)
ax.set_title('Figure 1 — error outside the training box'); plt.tight_layout(); plt.show()

## 5. The pricer bug — Business Snapshot 39.1

At 200 nodes the quadrature has not converged where the characteristic function
oscillates fastest: deep out of the money, short dated, high vol-of-vol. It returns
**negative call prices** — an impossible value any check would catch. The network
then smooths them into plausible volatilities, and the bug stops being detectable.

In [ ]:
hard = dict(S=100.0, K=175.0, T=0.12, r=R, q=Q, v0=0.15, theta=0.15,
            kappa=0.6, sigma_v=0.95, rho=-0.05)
for n in (200, 400, 600, 1200, 2400):
    print('%5d nodes  price %+.10e' % (n, heston_call(n_nodes=n, **hard)))
print('\nA negative price is impossible; a plausible implied volatility is not.')
print('Validate the labels before validating the fit, and validate them where')
print('they are hardest to compute rather than where they are easy.')

## 6. Differential learning — Example 5.1

Learn a one-year call price as a function of spot from **single-path** payoffs.
Each path also yields its pathwise delta $\mathbf{1}\{S_T>K\}\,S_T/S_0$ from the
same draw. §39.5's construction is Taylor augmentation: each path contributes its
own point *plus two neighbors implied by its slope* — adding **data**, not a
penalty term, so no second-order differentiation of the network is required.

In [ ]:
K1, T1, SIG1 = 100.0, 1.0, 0.20


def paths(n, seed):
    rng = np.random.default_rng(seed)
    S = rng.uniform(60, 160, n)                                   # random spots
    z = rng.standard_normal(n)
    ST = S * np.exp((R - 0.5 * SIG1**2) * T1 + SIG1 * np.sqrt(T1) * z)
    y = np.exp(-R * T1) * np.maximum(ST - K1, 0.0)                # noisy payoff
    dy = np.exp(-R * T1) * (ST > K1) * ST / S                     # pathwise delta
    return S, y, dy


def fit_1d(Sx, yx, seed=0, epochs=None):
    epochs = epochs or (150 if FAST else 400)
    m, s = Sx.mean(), Sx.std()
    net = MLP([1, 32, 32, 1], seed=seed)
    opt = Adam([w.shape for w in net.W] + [b.shape for b in net.b], lr=5e-3)
    X = ((Sx - m) / s).reshape(-1, 1); Y = yx.reshape(-1, 1)
    rng = np.random.default_rng(seed + 7)
    for _ in range(epochs):
        idx = rng.permutation(len(X))
        for a in range(0, len(idx), 256):
            j = idx[a:a + 256]
            out, cache = net.forward(X[j])
            dW, db, _ = net.backward(cache, 2.0 * (out - Y[j]) / len(j))
            opt.step(net.W + net.b, dW + db)
    return lambda q: net(((q - m) / s).reshape(-1, 1)).ravel()


grid = np.linspace(70, 150, 200)
truth = np.array([bs_call(g, K1, T1, R, Q, SIG1) for g in grid])
counts = [256, 1024, 4096] + ([] if FAST else [16384])
print('%8s %14s %18s %13s' % ('Paths', 'Plain RMSE', 'With derivatives', 'Improvement'))
for n in counts:
    S, y, dy = paths(n, 11)
    plain = fit_1d(S, y)
    h = 0.25 * S.std()                       # a fixed fraction of the input's sd
    Sa = np.concatenate([S, S + h, S - h])
    ya = np.concatenate([y, y + h * dy, y - h * dy])
    aug = fit_1d(Sa, ya)
    e0 = np.sqrt(((plain(grid) - truth) ** 2).mean())
    e1 = np.sqrt(((aug(grid) - truth) ** 2).mean())
    print('%8d %14.4f %18.4f %12.2fx' % (n, e0, e1, e0 / e1))

**The test §39.5 insists on.** Replace the derivative labels with noise. A correct
implementation degrades sharply; a smoothness penalty does not.

In [ ]:
S, y, dy = paths(4096, 11)
h = 0.25 * S.std()
rngn = np.random.default_rng(5)
for tag, d in (('true derivative labels', dy),
               ('labels replaced by noise', rngn.permutation(dy))):
    Sa = np.concatenate([S, S + h, S - h])
    ya = np.concatenate([y, y + h * d, y - h * d])
    f = fit_1d(Sa, ya)
    print('%-26s RMSE %.4f' % (tag, np.sqrt(((f(grid) - truth) ** 2).mean())))

## 7. Deep hedging — Example 7.1

Sell one at-the-money call, 30 daily rebalances, and minimize CVaR$_{95}$ of the
terminal P&L under proportional costs.

Two things from §39.6 are load-bearing in the code below:

* **Rockafellar–Uryasev.** $\mathrm{CVaR}_\alpha(L)=\min_w\{w+\frac{1}{1-\alpha}E[(L-w)^+]\}$,
  so $w$ is carried as one extra trained scalar and only tail paths get a gradient.
* **The recursion over dates.** The holding chosen at $t$ is an input at $t+1$, so
  the gradient accumulates backward through the schedule — `dL_ddelta` below is
  exactly that recursion, and it is why the current position must be an input.

In [ ]:
S0H, KH, TH, SIGH, NSTEP = 100.0, 100.0, 1.0 / 12, 0.20, 30
R_HEDGE = 0.0                    # the chapter's hedging examples use r = 0
DT = TH / NSTEP
PREMIUM = bs_call(S0H, KH, TH, R_HEDGE, Q, SIGH)
ALPHA = 0.95
print('Black-Scholes premium %.4f' % PREMIUM)


def make_paths(n, seed):
    rng = np.random.default_rng(seed)
    z = rng.standard_normal((n, NSTEP))
    logS = np.cumsum((R_HEDGE - 0.5 * SIGH**2) * DT + SIGH * np.sqrt(DT) * z, axis=1)
    return S0H * np.exp(np.hstack([np.zeros((n, 1)), logS]))       # (n, NSTEP+1)


def bs_delta(S, tau):
    tau = np.maximum(tau, 1e-8)
    d1 = (np.log(S / KH) + (R_HEDGE + 0.5 * SIGH**2) * tau) / (SIGH * np.sqrt(tau))
    return norm.cdf(d1)


def pnl_from_holdings(S, hold, cost):
    """Terminal loss L = -(premium + trading gains - fees - payoff)."""
    gains = (hold * np.diff(S, axis=1)).sum(axis=1)
    prev = np.hstack([np.zeros((len(S), 1)), hold[:, :-1]])
    fees = cost * (S[:, :-1] * np.abs(hold - prev)).sum(axis=1)
    fees += cost * S[:, -1] * np.abs(hold[:, -1])                  # unwind at expiry
    payoff = np.maximum(S[:, -1] - KH, 0.0)
    return -(PREMIUM + gains - fees - payoff), fees

In [ ]:
def cvar(L, alpha=ALPHA):
    w = np.quantile(L, alpha)
    return w + np.maximum(L - w, 0).mean() / (1 - alpha)


def run_benchmarks(S, cost):
    tau = TH - np.arange(NSTEP) * DT
    d = bs_delta(S[:, :-1], tau)
    L, fees = pnl_from_holdings(S, d, cost)
    out = {'Black-Scholes delta': (-L.mean(), L.std(), cvar(L), fees.mean())}
    best = None
    for w_band in np.arange(0.0, 0.30, 0.01):                      # tuned no-trade band
        h = np.zeros_like(d); cur = np.zeros(len(S))
        for t in range(NSTEP):
            tgt = d[:, t]
            cur = np.where(np.abs(tgt - cur) > w_band, tgt, cur)
            h[:, t] = cur
        Lb, fb = pnl_from_holdings(S, h, cost)
        c = cvar(Lb)
        if best is None or c < best[0]:
            best = (c, w_band, -Lb.mean(), Lb.std(), fb.mean())
    out['band, w = %.2f' % best[1]] = (best[2], best[3], best[0], best[4])
    return out

In [ ]:
def features(S, tau, prev):
    """Policy inputs scaled to order one: moneyness in tenths, time left as a
    fraction of the option's life, and the previous holding. Unscaled, the time
    input spans only 0 to 1/12 and the network cannot resolve it."""
    return np.column_stack([10 * (S / KH - 1), np.full(len(S), tau / TH), prev])

def train_policy(cost, seed=7, epochs=None, n_paths=None, lr=2e-3, batch=2048):
    """Policy delta_t = f(moneyness, time left, previous holding), CVaR objective."""
    epochs = epochs or (12 if FAST else 40)
    n_paths = n_paths or (40000 if FAST else 200000)
    S = make_paths(n_paths, seed)
    net = MLP([3, 32, 32, 1], seed=seed)
    opt = Adam([w.shape for w in net.W] + [b.shape for b in net.b], lr=lr)
    w_var = np.array([np.quantile(pnl_from_holdings(S, bs_delta(S[:, :-1], TH - np.arange(NSTEP) * DT), cost)[0], ALPHA)])
    opt_w = Adam([w_var.shape], lr=lr * 5)
    rng = np.random.default_rng(seed + 3)
    tau = TH - np.arange(NSTEP) * DT

    for ep in range(epochs):
        idx = rng.permutation(len(S))
        for a in range(0, len(idx), batch):
            P = S[idx[a:a + batch]]; n = len(P)
            # ---- forward through the schedule, caching every date -------------
            hold = np.zeros((n, NSTEP)); caches = []; prev = np.zeros(n)
            for t in range(NSTEP):
                x = features(P[:, t], tau[t], prev)
                out, cache = net.forward(x)
                prev = out.ravel(); hold[:, t] = prev; caches.append(cache)
            L, _ = pnl_from_holdings(P, hold, cost)
            # ---- Rockafellar-Uryasev: only the tail carries a gradient --------
            w = w_var[0]
            tail = (L > w).astype(float)
            dL = tail / ((1 - ALPHA) * n)
            gw = np.array([1.0 - tail.mean() / (1 - ALPHA)])
            # ---- backward through the dates (the recursion of 39.6) ----------
            gW = [np.zeros_like(w_) for w_ in net.W]
            gB = [np.zeros_like(b_) for b_ in net.b]
            carry = np.zeros(n)                        # d L / d delta_t from t+1
            for t in reversed(range(NSTEP)):
                dS = P[:, t + 1] - P[:, t]
                prev_h = hold[:, t - 1] if t > 0 else np.zeros(n)
                nxt_h = hold[:, t + 1] if t < NSTEP - 1 else np.zeros(n)
                sgn_now = np.sign(hold[:, t] - prev_h)
                sgn_nxt = np.sign(nxt_h - hold[:, t])
                dPi = dS - cost * P[:, t] * sgn_now + cost * P[:, t + 1] * sgn_nxt
                grad_t = -dL * dPi + carry            # L = -(...) hence the sign
                dWt, dBt, dx = net.backward(caches[t], grad_t.reshape(-1, 1))
                for i in range(len(gW)):
                    gW[i] += dWt[i]; gB[i] += dBt[i]
                carry = dx[:, 2]                       # sensitivity to prev holding
            opt.step(net.W + net.b, gW + gB)
            opt_w.step([w_var], [gw])
    return net


def evaluate(net, S, cost):
    n = len(S); tau = TH - np.arange(NSTEP) * DT
    hold = np.zeros((n, NSTEP)); prev = np.zeros(n)
    for t in range(NSTEP):
        x = features(S[:, t], tau[t], prev)
        prev = net(x).ravel(); hold[:, t] = prev
    L, fees = pnl_from_holdings(S, hold, cost)
    return (-L.mean(), L.std(), cvar(L), fees.mean()), hold

In [ ]:
S_eval = make_paths(40000 if FAST else 200000, 12345)
print('%5s  %-22s %9s %8s %10s %8s' % ('Cost', 'Strategy', 'Mean', 'Std dev', 'CVaR95', 'Fees'))
policies = {}
for cost_bp in (0, 10, 50):
    c = cost_bp / 1e4
    for name, v in run_benchmarks(S_eval, c).items():
        print('%4d  %-22s %9.4f %8.4f %10.4f %8.4f' % (cost_bp, name, *v))
    pol = train_policy(c, seed=7)
    stats, hold = evaluate(pol, S_eval, c)
    policies[cost_bp] = (pol, hold)
    print('%4d  %-22s %9.4f %8.4f %10.4f %8.4f' % (cost_bp, 'learned', *stats))
    print()

## 8. Seed dependence — Example 12.2 and Figure 4

Same code, same data, three seeds. **The performance reproduces and the strategy
does not** — which is why a surrogate's deliverable (a price) and a policy's
deliverable (a holding) need different governance.

In [ ]:
spots = np.array([95.0, 100.0, 105.0])
tau_mid = TH / 2
print('%6s %10s %9s %8s %8s %8s' % ('Seed', 'CVaR95', 'Fees', '95', '100', '105'))
curves = {}
for sd_ in (7, 17, 27):
    pol = train_policy(50 / 1e4, seed=sd_)
    stats, _ = evaluate(pol, S_eval, 50 / 1e4)
    x = features(spots, tau_mid, bs_delta(spots, tau_mid))
    h = pol(x).ravel()
    curves[sd_] = h
    print('%6d %10.4f %9.4f %8.4f %8.4f %8.4f' % (sd_, stats[2], stats[3], *h))
bsd = bs_delta(spots, tau_mid)
print('%6s %10s %9s %8.4f %8.4f %8.4f' % ('BS', '', '', *bsd))

fig, ax = plt.subplots(figsize=(5.4, 3.4))
for (sd_, h), st in zip(curves.items(), ['-o', '--s', ':^']):
    ax.plot(spots, h, st, color='#1f3b63', label='seed %d' % sd_, ms=4)
ax.plot(spots, bsd, '-.o', color='#b8860b', lw=2, ms=4, label='Black–Scholes delta')
ax.set_xlabel('Spot'); ax.set_ylabel('Holding'); ax.legend(fontsize=8)
ax.set_title('Figure 4 — seed dependence'); plt.tight_layout(); plt.show()

print('\nWhat reproduces is the shape: every seed sits above the delta at 95')
print('and below it at 105. The holding itself is an artifact of the seed.')